# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, inspect, and process the [FAIR² dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. All dataset elements, such as record sets and fields, are referenced by their Croissant schema `@id`s.

### Dataset Source

The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

We first load the dataset metadata and access its main attributes using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show basic info
print(f"Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Version: {metadata.version}")

## 2. Data Overview

We now review the record sets, their IDs, and available fields. All record sets and field names are referenced by their `@id` for reproducibility.


In [ ]:
# List all record sets and their fields by @id

print("Record Sets:")
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"  - {rs.id}: {rs.name}")
    print("    Fields (by @id):")
    for field in rs.fields:
        print(f"      - {field.id}: {field.name} ({field.data_type})")

## 3. Data Extraction

We can now load records from one or more record sets. Here, we'll extract all record sets defined in the schema, referencing their `@id`s dynamically. This ensures the code always points to the correct data, regardless of any schema changes.

Let's examine the columns (field `@id`s) for our main record set and show the first few records.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Load records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set '{record_set_id}' with shape: {df.shape}")

# Select the main record set (the one with most columns/records)
if record_set_ids:
    # For demonstration, select the largest
    main_record_set_id = max(record_set_ids, key=lambda k: dataframes[k].shape[1])
else:
    main_record_set_id = None

if main_record_set_id is not None:
    print(f"\nField (column) @ids in main record set '{main_record_set_id}':")
    print(list(dataframes[main_record_set_id].columns))
    print("\nFirst few records:")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in the schema.")

## 4. Exploratory Data Analysis (EDA)

Explore and process the data. Select a numeric field (referenced by `@id`) for filtering, normalization, and grouping. Adjust the `numeric_field_id` and `group_field_id` variables to reference the appropriate field `@id`s from the record set above.


In [ ]:
# Set the field @id for numeric and group operations
# Example: numeric_field_id = 'cr:coverage_percent', group_field_id = 'cr:group'

if main_record_set_id is not None:
    # Guess a numeric field (try to find one with float/int type columns)
    main_df = dataframes[main_record_set_id]
    numeric_candidates = main_df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    else:
        numeric_field_id = None

    if numeric_field_id:
        print(f"Using numeric field: {numeric_field_id}")
        threshold = main_df[numeric_field_id].quantile(0.5) if not main_df[numeric_field_id].isnull().all() else 0
        filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (median): {len(filtered_df)} rows")

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print("\nFirst few normalized values:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by a categorical field if present
        group_candidates = main_df.select_dtypes(include=['object']).columns.tolist()
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"\nGrouping by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            display(grouped_df.head())
        else:
            print("No categorical fields available for grouping.")
    else:
        print("No numeric field found for analysis.")
else:
    print("No dataframes loaded.")

## 5. Visualization

Visualize distributions or relationships for numeric fields. Adjust the field `@id` if necessary.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of selected numeric field
if main_record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouping field available, boxplot
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(12,4))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Numerical data is not available for visualization.")

## 6. Conclusion

In this notebook, we've demonstrated loading and accessing a rich biological dataset defined by a Croissant schema using `mlcroissant`. By referencing all data entities via their `@id`, our analysis remains reproducible and robust against schema changes. We've performed basic filtering, normalization, and visualization of the data. You can extend this workflow with additional processing tailored to your research objectives.